In [ ]:
from basic import TasnsfModel
from tokenizer.tokenizer import encode, decode
import torch
import json
from tqdm import tqdm
import random

device = "cuda:2"

In [ ]:
model_names = ["epoch_1_step_10000"]
models = {}

for m_name in model_names:
    model = TasnsfModel(vocab_size=25000, max_seq_len=1024, embd_dim=2048, num_head=16, num_layers=20).to(device)
    model.load_state_dict(torch.load(f"models/{m_name}.pth", map_location=device))
    model.eval()
    models[m_name] = model

In [ ]:
def apply_repet_penalty(logits, gen_tokens, repet_penalty):
    for token in set(gen_tokens):
        logit = logits[token]
        if logit > 0:
            logits[token] = logit / repet_penalty
        else:
            logits[token] = logit * repet_penalty
            
    return logits

In [ ]:
def generate(text, model=model, max_gen_len=128, temp=0.3, repet_penalty=1.2, printing=True):

    inp_tokens = encode(text)
    gen_tokens = []
    next_token = -1
    pad_token = encode("<pad>")[0]

    while next_token != pad_token and len(gen_tokens) < max_gen_len and not gen_tokens[-2:]==[10, 10]:
        tokens = inp_tokens + gen_tokens
        tokens = torch.tensor(tokens, dtype=torch.long, device=device).unsqueeze(0)
        out = model(tokens)
        
        logits = out[:, -1]
        logits = logits.squeeze(0)
        logits = apply_repet_penalty(logits, gen_tokens, repet_penalty)
        logits = logits / temp
        
        probs = torch.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, 1).item()
        
        gen_tokens.append(next_token)
        if printing:
            try:
                print(decode([next_token]), end="")
            except:
                pass

    return decode(gen_tokens)

In [ ]:
with open("../data/verse_data/verse_ids.json") as f:
    verse_ids = json.load(f)
    
print("Orignal verses: ", len(verse_ids))
print("Updating verse_ids...")

new_verse_ids = verse_ids.copy()
for id in tqdm(verse_ids):
    with open(f"../data/verse_data/{id}.json", "r") as f:
        v_data = json.load(f)
        
    new_verse_ids.extend([id] * len(v_data["misc_data"]))
print("Final verses: ", len(new_verse_ids))

verse_ids = new_verse_ids

In [ ]:
ids = random.sample(verse_ids, 1)
for id in ids:
    with open(f"../data/verse_data/{id}.json", "r") as f:
        v_data = json.load(f)
        
    parts = ["verse :\n" + v_data["text"], "reference :\n" + v_data["ref"]]

    random.shuffle(v_data["misc_data"])
    for m_data in v_data["misc_data"][:2]:
        parts.append(m_data["title"] + " :\n" + m_data["value"])
        
    random.shuffle(parts)
    text = "\n\n".join(parts[:int(2.1 + (random.random()*3))])

    orig_ans = text[text.rfind(" :\n") + 3:]
    text = text[:text.rfind(" :\n")+3]
    
    print("Input:\n", text)
    print("- "*20, "\n")
    print("Answer:\n", orig_ans)
    
    for m_name, model in models.items():
        print("\n", "- "*20, "\n")
        op = generate(text, model, printing=False)
        print(m_name, ":\n",op)